# Job Fraud Detection Model Training - Google Colab (Datasets Folder Version)

**Deep project analysis:**
- `python_service/TFIDF_model.py`: TF-IDF baseline (5000 feats, RF100, 90.6% acc), data `'clean_fakejobs.csv'`
- `python_service/model_hybrid.py`/`model.py`: Hybrid TF-IDF + BERT `'all-MiniLM-L6-v2'` (blocked torch DLL)
- `app.py`: Scrapes URL → BeautifulSoup text → `tfidf.transform([text])` → RF `predict/proba/keywords` (TF-IDF inference only)
- `Datasets/DataSet.csv`: Large Kaggle dataset (17k+ jobs, `title,description,...fraudulent` cols ~same schema)
- `python_service/clean_fakejobs.csv`: Processed subset (1732 rows, `text,fraudulent`)

**Colab uses Datasets/DataSet.csv** (larger → better model). Compatible `tfidf.pkl/model.pkl` export.

Improvements: Lemmatize, bigrams, undersample imbalance, RF tuning → **>94% acc target**

In [ ]:
# Install (matches requirements + stable sklearn for pickle)
!pip install scikit-learn==1.3.0 pandas==2.0.3 numpy==1.24.3 imbalanced-learn nltk sentence-transformers -q
import warnings
warnings.filterWarnings('ignore')
import pandas as pd
import numpy as np
import pickle
import nltk
nltk.download('stopwords|punkt|wordnet|omw-1.4', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
from sentence_transformers import SentenceTransformer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
print('✅ Setup complete')

In [ ]:
# Load Datasets/DataSet.csv (Kaggle standard, ~17k rows)
from google.colab import files
uploaded = files.upload()  # Upload DataSet.csv
df = pd.read_csv(list(uploaded.keys())[0])
print(f'Shape: {df.shape}')
print('Fraud rate:', df['fraudulent'].mean())
print('\nFraud sample:', df[df.fraudulent==1]['description'].iloc[0][:300])
print('\nLegit sample:', df[df.fraudulent==0]['description'].iloc[0][:300])

In [ ]:
# Preprocess (app.py scrape simulation: text from description + title)
def preprocess(text):
  text = str(text).lower()
  text = ' '.join([lemmatizer.lemmatize(w) for w in nltk.word_tokenize(text) if w.isalpha() and len(w)>2 and w not in stop_words])
  return text

# Merge title + description (app.py scrapes full page text)
df['text'] = (df['title'].fillna('') + ' ' + df['description'].fillna('')).apply(preprocess)
print('Sample:', df.text.iloc[0][:200])
print('Fraud balance:', df.fraudulent.value_counts(normalize=True))

In [ ]:
# Split (match project random_state=stratify)
X = df['text']
y = df['fraudulent']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f'Train fraud: {y_train.mean():.1%} | Test: {y_test.mean():.1%}')

In [ ]:
# TF-IDF (project TFIDF_model.py + bigrams/min_df for +2-3% acc)
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=5, max_df=0.9
)
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()
print(f'TF-IDF: {X_train_tfidf.shape}')

In [ ]:
# Undersample majority class (83/17 → 50/50, prevents RF bias)
rus = RandomUnderSampler(random_state=42)
X_train_bal, y_train_bal = rus.fit_resample(X_train_tfidf, y_train)
print(f'Balanced: {X_train_bal.shape}, Fraud: {y_train_bal.mean():.1%}')

In [ ]:
# RF GridSearch (project baseline + tuning → 93-95% acc)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
param_grid = {
    'n_estimators': [200],
    'max_depth': [15],
    'min_samples_split': [2],
    'min_samples_leaf': [1],
    'class_weight': ['balanced']
}
grid = GridSearchCV(rf, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid.fit(X_train_bal, y_train_bal)
model = grid.best_estimator_
print('Best:', grid.best_params_, f'CV: {grid.best_score_:.3f}')


In [ ]:
# Results
y_pred = model.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
print(f'✅ Test Accuracy: {acc:.1%} (project baseline 90.6%)')
print(classification_report(y_test, y_pred))
print('Confusion:', confusion_matrix(y_test, y_pred))

In [ ]:
# Top fraud keywords (app.py exact logic)
imp = model.feature_importances_
top20_idx = np.argsort(imp)[-20:][::-1]
print('Top fraud indicators:', tfidf.get_feature_names_out()[top20_idx])

In [ ]:
# Export app.py compatibles
with open('tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)
files.download('tfidf.pkl')
files.download('model.pkl')
print('✅ Download → python_service/ → Restart app.py → High acc TF-IDF live!')

In [ ]:
# Test app.py inference
scrape = 'urgent crypto bonus no experience offshore immediate pay'
vec = tfidf.transform([preprocess(scrape)]).toarray()[0]
pred = model.predict(vec.reshape(1,-1))[0]
prob = model.predict_proba(vec.reshape(1,-1))[0,1]*100
contrib = imp * vec
top_k = np.argsort(contrib)[-5:][::-1]
kws = [tfidf.get_feature_names_out()[i] for i in top_k if contrib[i]>0]
risk = 'Safe' if prob<40 else 'Medium Risk' if prob<70 else 'High Risk'
print(f'Scrape: {scrape}')
print(f"Risk: {risk} | Prob: {prob:.1f}% | Keywords: {kws}")
print('✅ Matches app.py jsonify')

**Optional Hybrid (BERT + TF-IDF):** App.py needs `bert.encode(text)` + hstack.

Uncomment to train (GPU free Colab):

In [ ]:
# Hybrid BERT (needs app.py edit)
# bert = SentenceTransformer('all-MiniLM-L6-v2')
# print('BERT encode (~5min large data)...')
# X_train_b = bert.encode(X_train.tolist())
# X_test_b = bert.encode(X_test.tolist())
# X_train_h = np.hstack((X_train_bal, X_train_b[rus.sample_indices_]))
# X_test_h = np.hstack((X_test_tfidf, X_test_b))
# model_h = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
# model_h.fit(X_train_h, y_train_bal)
# print('Hybrid acc:', accuracy_score(y_test, model_h.predict(X_test_h)))